# Sleep Disorder model rebuild

Compact, product-oriented notebook for the `sleep_disorder` target.

Goals:
- keep feature sets short enough for a quiz
- always include `age_years`, `gender`, `med_count`
- test whether standard Check-up-35 style labs help
- use tuned Logistic Regression with scaling
- compare compact runs fairly on the same train/test split

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import re
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, RepeatedStratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

In [ ]:
DATA_PATH = Path("../data/processed/nhanes_merged_adults_final.csv")
OUTPUT_DIR = Path("../data/processed/model_outputs_sleep_disorder_rebuild")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "sleep_disorder"
RANDOM_STATE = 42
TEST_SIZE = 0.20

print("DATA_PATH:", DATA_PATH.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())

In [ ]:
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Target present:", TARGET in df.columns)
print("First 12 columns:", df.columns[:12].tolist())

In [ ]:
assert TARGET in df.columns, f"{TARGET} not found in dataframe."

print(df[TARGET].value_counts(dropna=False).sort_index())
print()
print("Target prevalence:")
print(df[TARGET].value_counts(normalize=True, dropna=False).sort_index().round(4))

## Clean questionnaire-style special codes

NHANES often uses 7 / 9 / 77 / 99 style values for "refused", "don't know", etc.
We convert common questionnaire-like columns to `NaN` before modeling.

In [ ]:
SPECIAL_MISSING_CODES = {
    7, 9,
    77, 99,
    777, 999,
    7777, 9999,
    77777, 99999,
}

QUESTIONNAIRE_LIKE_PREFIXES = (
    "slq", "sld", "dpq", "paq", "huq", "mcq", "bpq", "kiq", "cdq", "smq", "alq", "whq"
)

def replace_special_codes_with_nan(series: pd.Series) -> pd.Series:
    if not pd.api.types.is_numeric_dtype(series):
        return series
    return series.replace(list(SPECIAL_MISSING_CODES), np.nan)

questionnaire_like_cols = [
    c for c in df.columns
    if c.lower().startswith(QUESTIONNAIRE_LIKE_PREFIXES)
]

print("Questionnaire-like columns to clean:", len(questionnaire_like_cols))
df[questionnaire_like_cols] = df[questionnaire_like_cols].apply(replace_special_codes_with_nan)

In [ ]:
overview = pd.DataFrame({
    "feature": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "missing_pct": (df.isna().mean() * 100).round(2).values,
    "n_unique": [df[c].nunique(dropna=True) for c in df.columns],
}).sort_values(["missing_pct", "n_unique"], ascending=[False, True])

overview.head(30)

## Compact feature design

We deliberately keep the product-facing feature sets short.
For sleep disorder, standard Check-up-35 labs are not expected to be as strong as in
metabolically-driven conditions, but we test them as optional context.

In [ ]:
DEMO_COLS = [
    "age_years",
    "gender",
]

MED_COLS = [
    "med_count",
]

SCREENING_LAB_COLS = [
    "fasting_glucose_mg_dl",
    "hdl_cholesterol_mg_dl",
    "total_cholesterol_mg_dl",
    "triglycerides_mg_dl",
    "uacr_mg_g",
]

# Optional richer context labs if available. These are not necessarily standard Check-up-35 uploads.
AUGMENTED_LAB_COLS = [
    "serum_creatinine_mg_dl",
    "bmi",
]

SLEEP_QUIZ_COLS = [
    "slq030___how_often_do_you_snore?",
    "sld012___sleep_hours___weekdays_or_workdays",
    "sld013___sleep_hours___weekends",
    "dpq040___feeling_tired_or_having_little_energy",
    "bpq020___ever_told_you_had_high_blood_pressure",
    "cdq010___shortness_of_breath_on_stairs/inclines",
    "paq665___moderate_recreational_activities",
    "alq111___ever_had_a_drink_of_any_kind_of_alcohol",
    "mcq010___ever_been_told_you_have_asthma",
]

def existing_cols(cols, frame):
    return [c for c in cols if c in frame.columns]

def missing_cols(cols, frame):
    return [c for c in cols if c not in frame.columns]

def dedupe_keep_order(seq):
    seen = set()
    out = []
    for x in seq:
        if x not in seen:
            out.append(x)
            seen.add(x)
    return out

print("Available sleep quiz cols:", existing_cols(SLEEP_QUIZ_COLS, df))
print("Missing sleep quiz cols:", missing_cols(SLEEP_QUIZ_COLS, df))

print("\nAvailable demo cols:", existing_cols(DEMO_COLS, df))
print("Available med cols:", existing_cols(MED_COLS, df))
print("Available screening labs:", existing_cols(SCREENING_LAB_COLS, df))
print("Available augmented labs:", existing_cols(AUGMENTED_LAB_COLS, df))

In [ ]:
feature_sets = {
    "compact_quiz_only": dedupe_keep_order(
        existing_cols(SLEEP_QUIZ_COLS, df)
    ),
    "compact_quiz_demo_med": dedupe_keep_order(
        existing_cols(SLEEP_QUIZ_COLS, df)
        + existing_cols(DEMO_COLS, df)
        + existing_cols(MED_COLS, df)
    ),
    "compact_quiz_demo_med_screening_labs": dedupe_keep_order(
        existing_cols(SLEEP_QUIZ_COLS, df)
        + existing_cols(DEMO_COLS, df)
        + existing_cols(MED_COLS, df)
        + existing_cols(SCREENING_LAB_COLS, df)
    ),
    "compact_quiz_demo_med_augmented_labs": dedupe_keep_order(
        existing_cols(SLEEP_QUIZ_COLS, df)
        + existing_cols(DEMO_COLS, df)
        + existing_cols(MED_COLS, df)
        + existing_cols(SCREENING_LAB_COLS, df)
        + existing_cols(AUGMENTED_LAB_COLS, df)
    ),
}

pd.DataFrame({
    "run": list(feature_sets.keys()),
    "n_features": [len(v) for v in feature_sets.values()],
    "features": [", ".join(v) for v in feature_sets.values()],
})

## Shared split for fair comparisons

In [ ]:
all_model_features = dedupe_keep_order(sum(feature_sets.values(), []))

model_df = df[[TARGET] + all_model_features].copy()
mask = model_df[TARGET].notna()

X_all = model_df.loc[mask, all_model_features].copy()
y_all = model_df.loc[mask, TARGET].astype(int).copy()

X_train_all, X_test_all, y_train, y_test = train_test_split(
    X_all,
    y_all,
    test_size=TEST_SIZE,
    stratify=y_all,
    random_state=RANDOM_STATE,
)

print("Train shape:", X_train_all.shape)
print("Test shape:", X_test_all.shape)
print("Train target distribution:")
print(y_train.value_counts(normalize=True).round(4))

In [ ]:
def infer_feature_type(series: pd.Series) -> str:
    if pd.api.types.is_numeric_dtype(series):
        nunique = series.nunique(dropna=True)
        if nunique <= 10:
            return "categorical"
        return "numeric"
    return "categorical"

def split_feature_types(X_frame: pd.DataFrame, feature_list):
    numeric_features = []
    categorical_features = []
    for col in feature_list:
        if infer_feature_type(X_frame[col]) == "numeric":
            numeric_features.append(col)
        else:
            categorical_features.append(col)
    return numeric_features, categorical_features

def build_logreg_preprocessor(numeric_features, categorical_features):
    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    return ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ],
        remainder="drop"
    )

def evaluate_predictions(y_true, y_proba, threshold=0.50):
    y_pred = (y_proba >= threshold).astype(int)
    return {
        "roc_auc": roc_auc_score(y_true, y_proba),
        "avg_precision": average_precision_score(y_true, y_proba),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "confusion_matrix": confusion_matrix(y_true, y_pred),
        "classification_report": classification_report(y_true, y_pred, zero_division=0),
    }

def run_tuned_logreg(feature_list, run_name):
    X_train = X_train_all[feature_list].copy()
    X_test = X_test_all[feature_list].copy()

    numeric_features, categorical_features = split_feature_types(X_train, feature_list)
    preprocessor = build_logreg_preprocessor(numeric_features, categorical_features)

    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            max_iter=5000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ])

    param_grid = [
        {
            "model__solver": ["liblinear"],
            "model__penalty": ["l1", "l2"],
            "model__C": [0.01, 0.1, 1, 3, 10],
        },
        {
            "model__solver": ["saga"],
            "model__penalty": ["l1", "l2"],
            "model__C": [0.01, 0.1, 1, 3, 10],
        },
    ]

    cv = RepeatedStratifiedKFold(
        n_splits=5,
        n_repeats=2,
        random_state=RANDOM_STATE
    )

    grid = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        scoring="average_precision",
        cv=cv,
        n_jobs=-1,
        verbose=1,
        refit=True
    )

    grid.fit(X_train, y_train)

    best_estimator = grid.best_estimator_
    y_test_proba = best_estimator.predict_proba(X_test)[:, 1]

    metrics = evaluate_predictions(y_test, y_test_proba, threshold=0.50)

    return {
        "run": run_name,
        "feature_list": feature_list,
        "n_features": len(feature_list),
        "numeric_features": numeric_features,
        "categorical_features": categorical_features,
        "best_estimator": best_estimator,
        "best_params": grid.best_params_,
        "cv_best_score_avg_precision": grid.best_score_,
        "y_test_true": y_test.copy(),
        "y_test_proba": y_test_proba,
        **metrics,
    }

In [ ]:
all_results = {}

for run_name, feature_list in feature_sets.items():
    print(f"\n=== Running {run_name} ===")
    all_results[run_name] = run_tuned_logreg(feature_list, run_name)

In [ ]:
comparison_df = pd.DataFrame([
    {
        "run": run_name,
        "n_features": result["n_features"],
        "cv_best_score_avg_precision": round(result["cv_best_score_avg_precision"], 4),
        "roc_auc": round(result["roc_auc"], 4),
        "avg_precision": round(result["avg_precision"], 4),
        "precision": round(result["precision"], 4),
        "recall": round(result["recall"], 4),
        "f1": round(result["f1"], 4),
        "best_params": json.dumps(result["best_params"]),
    }
    for run_name, result in all_results.items()
]).sort_values(["avg_precision", "roc_auc"], ascending=False).reset_index(drop=True)

comparison_df

## Choose preferred product model

Metrics may favor one run, but the best product model can still be a different one.
Default below is the screening-lab variant because it stays close to a standard lab upload workflow.

In [ ]:
auto_best_run = comparison_df.iloc[0]["run"]
auto_best_result = all_results[auto_best_run]

selected_run = "compact_quiz_demo_med_screening_labs"
selected_result = all_results[selected_run]

print("Auto-best run:", auto_best_run)
print("Selected run:", selected_run)
print("Best params:", selected_result["best_params"])
print("ROC-AUC:", round(selected_result["roc_auc"], 4))
print("Average Precision:", round(selected_result["avg_precision"], 4))
print("Precision:", round(selected_result["precision"], 4))
print("Recall:", round(selected_result["recall"], 4))
print("F1:", round(selected_result["f1"], 4))
print()
print("Confusion Matrix:")
print(selected_result["confusion_matrix"])
print()
print(selected_result["classification_report"])

## Threshold scan

In [ ]:
threshold_rows = []
y_true = selected_result["y_test_true"]
y_proba = selected_result["y_test_proba"]

roc_auc = roc_auc_score(y_true, y_proba)

for thr in np.arange(0.10, 0.91, 0.05):
    metrics = evaluate_predictions(y_true, y_proba, threshold=float(thr))
    threshold_rows.append({
        "threshold": round(float(thr), 2),
        "roc_auc": round(roc_auc, 4),
        "precision": round(metrics["precision"], 4),
        "recall": round(metrics["recall"], 4),
        "f1": round(metrics["f1"], 4),
    })

threshold_df = pd.DataFrame(threshold_rows)
threshold_df

## Coefficients in a readable form

In [ ]:
FEATURE_LABEL_MAP = {
    "age_years": "Age (years)",
    "gender": "Gender",
    "med_count": "Medication count",
    "fasting_glucose_mg_dl": "Fasting glucose",
    "hdl_cholesterol_mg_dl": "HDL cholesterol",
    "total_cholesterol_mg_dl": "Total cholesterol",
    "triglycerides_mg_dl": "Triglycerides",
    "uacr_mg_g": "Urine albumin-creatinine ratio (UACR)",
    "serum_creatinine_mg_dl": "Serum creatinine",
    "bmi": "BMI",
    "slq030___how_often_do_you_snore?": "How often do you snore?",
    "sld012___sleep_hours___weekdays_or_workdays": "Sleep hours on weekdays",
    "sld013___sleep_hours___weekends": "Sleep hours on weekends",
    "dpq040___feeling_tired_or_having_little_energy": "Feeling tired / little energy",
    "bpq020___ever_told_you_had_high_blood_pressure": "Ever told you had high blood pressure",
    "cdq010___shortness_of_breath_on_stairs/inclines": "Shortness of breath on stairs",
    "paq665___moderate_recreational_activities": "Moderate recreational activities",
    "alq111___ever_had_a_drink_of_any_kind_of_alcohol": "Ever had any alcohol",
    "mcq010___ever_been_told_you_have_asthma": "Ever told you had asthma",
}

VALUE_LABEL_MAP = {
    "gender": {
        "Male": "Male",
        "Female": "Female",
    },
    "bpq020___ever_told_you_had_high_blood_pressure": {
        "1.0": "Yes",
        "2.0": "No",
    },
    "cdq010___shortness_of_breath_on_stairs/inclines": {
        "1.0": "Yes",
        "2.0": "No",
    },
    "paq665___moderate_recreational_activities": {
        "1.0": "Yes",
        "2.0": "No",
    },
    "alq111___ever_had_a_drink_of_any_kind_of_alcohol": {
        "1.0": "Yes",
        "2.0": "No",
    },
    "mcq010___ever_been_told_you_have_asthma": {
        "1.0": "Yes",
        "2.0": "No",
    },
}

def prettify_transformed_feature_name(name: str) -> str:
    if name.startswith("num__missingindicator_"):
        raw = name.replace("num__missingindicator_", "")
        pretty_raw = FEATURE_LABEL_MAP.get(raw, raw)
        return f"{pretty_raw} missing"

    if name.startswith("num__"):
        raw = name.replace("num__", "")
        return FEATURE_LABEL_MAP.get(raw, raw)

    if name.startswith("cat__"):
        raw = name.replace("cat__", "")
        if "_" in raw:
            feature_part, value_part = raw.rsplit("_", 1)
            pretty_feature = FEATURE_LABEL_MAP.get(feature_part, feature_part)
            pretty_value = VALUE_LABEL_MAP.get(feature_part, {}).get(value_part, value_part)
            return f"{pretty_feature}: {pretty_value}"
        return FEATURE_LABEL_MAP.get(raw, raw)

    return name

best_estimator = selected_result["best_estimator"]
preprocessor = best_estimator.named_steps["preprocessor"]
model = best_estimator.named_steps["model"]

feature_names = preprocessor.get_feature_names_out()

coef_df_raw = pd.DataFrame({
    "transformed_feature": feature_names,
    "coefficient": model.coef_[0],
})
coef_df_raw["abs_coefficient"] = coef_df_raw["coefficient"].abs()
coef_df_raw = coef_df_raw.sort_values("abs_coefficient", ascending=False).reset_index(drop=True)

coef_df_pretty = coef_df_raw.copy()
coef_df_pretty["pretty_feature"] = coef_df_pretty["transformed_feature"].apply(prettify_transformed_feature_name)
coef_df_pretty = coef_df_pretty[
    ["pretty_feature", "coefficient", "abs_coefficient", "transformed_feature"]
]

coef_df_nonzero = coef_df_pretty[coef_df_pretty["coefficient"] != 0].copy()
coef_df_nonzero = coef_df_nonzero.sort_values("abs_coefficient", ascending=False).reset_index(drop=True)

coef_df_nonzero.head(40)

## Save reports

In [ ]:
comparison_path = OUTPUT_DIR / "sleep_disorder_compact_model_comparison.csv"
threshold_path = OUTPUT_DIR / f"{selected_run}_threshold_scan.csv"
coef_path = OUTPUT_DIR / f"{selected_run}_coefficients_pretty.csv"

comparison_df.to_csv(comparison_path, index=False)
threshold_df.to_csv(threshold_path, index=False)
coef_df_pretty.to_csv(coef_path, index=False)

print("Saved:")
print(comparison_path.resolve())
print(threshold_path.resolve())
print(coef_path.resolve())

## Save model package and metadata

Update `threshold` after deciding on the operating point you want in product.

In [ ]:
disease = "sleep_disorder"
threshold = 0.50  # change if you choose a different operating point
threshold_label = str(threshold).replace(".", "")

feature_names = feature_sets[selected_run]
model_package = {
    "model": selected_result["best_estimator"],
    "threshold": threshold,
    "features": feature_names,
    "model_name": selected_run,
    "target": TARGET,
    "disease": disease,
}

MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

joblib_path = MODELS_DIR / f"{disease}_{selected_run}_threshold_{threshold_label}.joblib"
joblib.dump(model_package, joblib_path)

print("Joblib saved to:", joblib_path.resolve())
print("Stored features:", feature_names)

In [ ]:
metadata = {
    "model_name": selected_run,
    "target": TARGET,
    "disease": disease,
    "threshold": threshold,
    "features": feature_names,
    "n_features": len(feature_names),
    "metrics": {
        "roc_auc": float(selected_result["roc_auc"]),
        "avg_precision": float(selected_result["avg_precision"]),
        "precision": float(selected_result["precision"]),
        "recall": float(selected_result["recall"]),
        "f1": float(selected_result["f1"]),
    },
    "best_params": selected_result["best_params"],
    "created_at": datetime.now(timezone.utc).isoformat(),
    "notes": "Compact sleep disorder model using quiz, demographics, medication count, and optional standard screening labs.",
}

metadata_path = MODELS_DIR / f"{disease}_{selected_run}_threshold_{threshold_label}.metadata.json"

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("Metadata JSON saved to:", metadata_path.resolve())

In [ ]:
loaded_package = joblib.load(joblib_path)
with open(metadata_path, "r", encoding="utf-8") as f:
    loaded_metadata = json.load(f)

print("Loaded package keys:", loaded_package.keys())
print("Loaded model name:", loaded_package["model_name"])
print("Loaded threshold:", loaded_package["threshold"])
print("Loaded feature count:", len(loaded_package["features"]))
print("Loaded metadata disease:", loaded_metadata["disease"])

## RETRAIN v2 — Normalized Data Pipeline

### v2 Changes vs v1

| # | Change | v1 | v2 |
|---|--------|----|----|
| 1 | Data source | `nhanes_merged_adults_final.csv` (raw) | `nhanes_merged_adults_final_normalized.csv` (z-scored) |
| 2 | Scaling | `StandardScaler` in ColumnTransformer | Removed — data pre-normalized |
| 3 | Missingness | `SimpleImputer` no indicator | `SimpleImputer(median, add_indicator=True)` |
| 4 | Gender encoding | Raw string `gender` column | `gender_female` binary (1=Female, 0=Male) |
| 5 | Evaluation | Single 80/20 split + GridSearchCV | 5-fold Stratified CV |
| 6 | Model output | Dict wrapping `{model, threshold, features}` | Plain `.joblib` pipeline |
| 7 | Threshold | Per-model 0.50 in dict | Global 0.35 via model_runner |
| 8 | Leakage removed | slq050 not in features (good) | Confirmed + slq040 excluded |

**Leakage audit:**
- `slq050___ever_told_doctor_had_trouble_sleeping?` — target component (used to define label), excluded from features
- `slq040___how_often_do_you_snort_or_stop_breathing` — target component, excluded
- `slq030___how_often_do_you_snore?` — symptom proxy (NOT in target definition), kept as feature

**Experiments:**
- **v2a** CSV features (17, slq050 removed)
- **v2b** All 79 roadmap features
- **v2c** L1-selected features
- **v2d** Comparison table + best model saved

In [1]:
# ── v2 Setup ─────────────────────────────────────────────────────────────
import os, warnings, json as _json, joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             recall_score, precision_score, f1_score, roc_curve)

warnings.filterwarnings('ignore')

DATA_V2       = '../data/processed/nhanes_merged_adults_final_normalized.csv'
MODEL_OUT_DIR = '../models_normalized'
TARGET_COL    = 'sleep_disorder'
RANDOM_STATE  = 42
CV_FOLDS      = 5
THRESHOLD     = 0.35

os.makedirs(MODEL_OUT_DIR, exist_ok=True)

df = pd.read_csv(DATA_V2, low_memory=False)
df['gender_female'] = (df['gender'] == 'Female').astype(float)

print(f"Dataset: {df.shape[0]:,} rows")
print(f"Target distribution:")
print(df[TARGET_COL].value_counts(dropna=False))
print(f"Prevalence: {df[TARGET_COL].mean():.1%}")

Dataset: 7,437 rows
Target distribution:
sleep_disorder
0    5034
1    2403
Name: count, dtype: int64
Prevalence: 32.3%


In [2]:
# ── v2 Feature lists ─────────────────────────────────────────────────────
# Source: roadmap CSV (sleep_disorder==1 rows)
# LEAKAGE REMOVED:
#   slq050___ever_told_doctor_had_trouble_sleeping?  <- target component (SLQ050 defines label)
#   slq040___how_often_do_you_snort_or_stop_breathing <- target component (SLQ040 defines label)
# KEPT (symptom proxy, not in target definition):
#   slq030___how_often_do_you_snore?

FEATURES_CSV_17 = [
    # Demographics
    'gender_female',
    'age_years',
    # Labs
    'hdl_cholesterol_mg_dl',
    'triglycerides_mg_dl',
    'total_cholesterol_mg_dl',
    'fasting_glucose_mg_dl',
    'uacr_mg_g',
    # Medication count
    'med_count',
    # Questionnaire
    'slq030___how_often_do_you_snore?',
    'sld012___sleep_hours___weekdays_or_workdays',
    'sld013___sleep_hours___weekends',
    'dpq040___feeling_tired_or_having_little_energy',
    'bpq020___ever_told_you_had_high_blood_pressure',
    'cdq010___shortness_of_breath_on_stairs/inclines',
    'paq665___moderate_recreational_activities',
    'alq111___ever_had_a_drink_of_any_kind_of_alcohol',
    'mcq010___ever_been_told_you_have_asthma',
]

# Full roadmap: all 79 usable features (same set as thyroid, minus sleep-specific leakage)
LEAKAGE_COLS = {
    'slq050___ever_told_doctor_had_trouble_sleeping?',   # target component
    'slq040___how_often_do_you_snort_or_stop_breathing', # target component
    TARGET_COL,
}

# Load roadmap CSV to build full feature list
import sys
sys.path.insert(0, '..')
csv_path = '/Users/annaesakova/Downloads/HalfFull roadmap - diseases VS features (1).csv'
df_map = pd.read_csv(csv_path, header=1)
df['education_ord'] = df['education'].map({
    'Less than 9th grade':1,'9-11th grade':2,'High school diploma/GED':3,
    'Some college / AA':4,'College graduate or above':5})
df_map.loc[df_map['canonical_feature']=='gender','norm_col'] = 'gender_female'
df_map.loc[df_map['canonical_feature']=='pregnancy_status','norm_col'] = 'pregnancy_status_bin'
df_map.loc[df_map['canonical_feature']=='education','norm_col'] = 'education_ord'
df['pregnancy_status_bin'] = df['pregnancy_status'].map({'Yes, pregnant':1.0,'Not pregnant':0.0})

norm_cols = set(df.columns)
def find_col(row):
    for c in [str(row['canonical_feature']).strip(),
              str(row.get('mapped_dataset_column','')).strip()
             ] + [a.strip() for a in str(row.get('source_feature_names','')).split('|')]:
        if c in norm_cols: return c
    return None
df_map['norm_col'] = df_map.apply(find_col, axis=1)
df_map.loc[df_map['canonical_feature']=='gender','norm_col'] = 'gender_female'
df_map.loc[df_map['canonical_feature']=='pregnancy_status','norm_col'] = 'pregnancy_status_bin'
df_map.loc[df_map['canonical_feature']=='education','norm_col'] = 'education_ord'

usable = df_map[
    df_map['norm_col'].notna() &
    (df_map['feature_type'] != 'unresolved-alias') &
    ~df_map['norm_col'].isin(LEAKAGE_COLS)
].drop_duplicates('norm_col')

FEATURES_ALL = [f for f in usable['norm_col'].tolist() if df[f].dtype != object]
print(f"CSV-17 features: {len(FEATURES_CSV_17)}")
print(f"All-roadmap features (no leakage, no string): {len(FEATURES_ALL)}")
missing = [f for f in FEATURES_CSV_17 if f not in df.columns]
print(f"Missing from normalized file: {missing or 'none'}")

CSV-17 features: 17
All-roadmap features (no leakage, no string): 78
Missing from normalized file: none


In [3]:
# ── v2 Pipeline + CV helper ──────────────────────────────────────────────
def build_pipe(penalty='l2', C=1.0):
    return Pipeline([
        ('imp', SimpleImputer(strategy='median', add_indicator=True)),
        ('clf', LogisticRegression(
            penalty=penalty, C=C, class_weight='balanced', max_iter=2000,
            solver='lbfgs' if penalty=='l2' else 'liblinear',
            random_state=RANDOM_STATE)),
    ])

def run_cv(feat, label, penalty='l2', C=1.0):
    df_c = df[feat+[TARGET_COL]].dropna(subset=[TARGET_COL])
    X = df_c[feat]; y = df_c[TARGET_COL].values.astype(int)
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    aucs, aps, recs, f1s = [], [], [], []
    for tr, te in cv.split(X, y):
        p = build_pipe(penalty, C)
        p.fit(X.iloc[tr], y[tr])
        prob = p.predict_proba(X.iloc[te])[:,1]
        pred = (prob >= THRESHOLD).astype(int)
        aucs.append(roc_auc_score(y[te], prob))
        aps.append(average_precision_score(y[te], prob))
        recs.append(recall_score(y[te], pred, zero_division=0))
        f1s.append(f1_score(y[te], pred, zero_division=0))
    print(f"  [{label:34s}] AUC={np.mean(aucs):.4f}+/-{np.std(aucs):.4f} "
          f"AP={np.mean(aps):.4f} Recall={np.mean(recs):.3f} F1={np.mean(f1s):.3f} "
          f"({len(feat)} feat)")
    return dict(label=label, n_feat=len(feat),
                auc=np.mean(aucs), auc_std=np.std(aucs),
                ap=np.mean(aps), recall=np.mean(recs), f1=np.mean(f1s)), aucs

print("Pipeline ready. Running experiments...")
print(f"CV: {CV_FOLDS}-fold stratified | threshold: {THRESHOLD}")

Pipeline ready. Running experiments...
CV: 5-fold stratified | threshold: 0.35


In [4]:
# ── v2a  CSV features (17, slq050 leakage removed) ───────────────────────
res_17,   folds_17   = run_cv(FEATURES_CSV_17, 'csv-17-L2',            'l2', 1.0)

  [csv-17-L2                         ] AUC=0.7623+/-0.0090 AP=0.6274 Recall=0.841 F1=0.577 (17 feat)


In [5]:
# ── v2b  All roadmap features (L2 + L1 variants) ─────────────────────────
res_all_l2,  folds_all_l2  = run_cv(FEATURES_ALL, 'all-roadmap L2 C=1',    'l2', 1.0)
res_all_l1a, folds_all_l1a = run_cv(FEATURES_ALL, 'all-roadmap L1 C=0.1',  'l1', 0.1)
res_all_l1b, folds_all_l1b = run_cv(FEATURES_ALL, 'all-roadmap L1 C=0.01', 'l1', 0.01)

  [all-roadmap L2 C=1                ] AUC=0.7820+/-0.0075 AP=0.6494 Recall=0.831 F1=0.602 (78 feat)
  [all-roadmap L1 C=0.1              ] AUC=0.7840+/-0.0099 AP=0.6508 Recall=0.841 F1=0.599 (78 feat)
  [all-roadmap L1 C=0.01             ] AUC=0.7713+/-0.0139 AP=0.6344 Recall=0.883 F1=0.567 (78 feat)


In [8]:
# ── v2c  L1 feature selection — which features survive? ──────────────────
df_c = df[FEATURES_ALL+[TARGET_COL]].dropna(subset=[TARGET_COL])
X = df_c[FEATURES_ALL]; y = df_c[TARGET_COL].values.astype(int)
p = build_pipe('l1', 0.1); p.fit(X, y)
imp = p.named_steps['imp']; clf = p.named_steps['clf']
all_names = list(FEATURES_ALL) + [f'miss__{FEATURES_ALL[i]}' for i in imp.indicator_.features_]
coefs = clf.coef_[0]
nonzero = sorted([(all_names[i], coefs[i]) for i in range(len(coefs)) if abs(coefs[i]) > 1e-6],
                  key=lambda x: abs(x[1]), reverse=True)
L1_SELECTED = [n for n,c in nonzero if not n.startswith('miss__')]
print(f"L1 (C=0.1) non-zero features: {len(nonzero)}  |  base features: {len(L1_SELECTED)}")
print(f" {'Feature':55s} {'Coef':>7}  Note")
print("  "+"-"*75)
for name, c in nonzero[:30]:
    new_flag = " << NEW" if (not name.startswith('miss__') and name not in FEATURES_CSV_17) else ""
    print(f"  {name:55s} {c:+.4f}{new_flag}")
if len(nonzero) > 30: print(f"  ... and {len(nonzero)-30} more")

L1 (C=0.1) non-zero features: 72  |  base features: 52
 Feature                                                    Coef  Note
  ---------------------------------------------------------------------------
  miss__mcq195___which_type_of_arthritis_was_it?          -0.5597
  med_count                                               +0.4959
  dpq040___feeling_tired_or_having_little_energy          +0.4564
  diq070___take_diabetic_pills_to_lower_blood_sugar       +0.4354 << NEW
  miss__alq151___ever_have_4/5_or_more_drinks_every_day?  -0.3347
  miss__mcq540___ever_seen_a_dr_about_this_pain           -0.2887
  mcq080___doctor_ever_said_you_were_overweight           -0.2316 << NEW
  miss__ocq180___hours_worked_last_week_in_total_all_jobs +0.2266
  cdq010___shortness_of_breath_on_stairs/inclines         -0.1948
  miss__mcq040___had_asthma_attack_in_past_year           -0.1776
  diq050___taking_insulin_now                             +0.1689 << NEW
  paq620___moderate_work_activity                

In [9]:
# ── v2c cont. — CV on L1-selected features only ─────────────────────────
res_l1sel, folds_l1sel = run_cv(L1_SELECTED, f'L1-selected-{len(L1_SELECTED)} L2', 'l2', 1.0)

  [L1-selected-52 L2                 ] AUC=0.7816+/-0.0086 AP=0.6493 Recall=0.831 F1=0.601 (52 feat)


In [10]:
# ── v2d  Results table + AUC box plot ────────────────────────────────────
all_res = [res_17, res_all_l2, res_all_l1a, res_all_l1b, res_l1sel]
rows = [{'Model': r['label'], 'Feat': r['n_feat'],
         'AUC': f"{r['auc']:.4f}+/-{r['auc_std']:.4f}",
         'AvgPrec': f"{r['ap']:.4f}",
         'Recall@.35': f"{r['recall']:.3f}",
         'F1@.35': f"{r['f1']:.3f}"} for r in all_res]
print("=== 5-fold CV Results ===")
print(pd.DataFrame(rows).to_string(index=False))

all_folds  = [folds_17, folds_all_l2, folds_all_l1a, folds_all_l1b, folds_l1sel]
box_labels = ['csv-17', 'all-79-L2', 'all-79-L1(.1)', 'all-79-L1(.01)', f'L1sel-{len(L1_SELECTED)}']
colors = ['#2196F3','#FF9800','#FF5722','#E91E63','#4CAF50']

fig, ax = plt.subplots(figsize=(9, 4))
bp = ax.boxplot(all_folds, patch_artist=True, widths=0.5,
                medianprops=dict(color='black', linewidth=2))
for patch, col in zip(bp['boxes'], colors):
    patch.set_facecolor(col); patch.set_alpha(0.7)
ax.set_xticks(range(1, len(box_labels)+1))
ax.set_xticklabels(box_labels, fontsize=9)
ax.set_ylabel('ROC-AUC (5-fold CV)')
ax.set_title('Sleep Disorder v2 — ROC-AUC comparison')
ax.grid(axis='y', alpha=0.4)
plt.tight_layout(); plt.show()

=== 5-fold CV Results ===
                Model  Feat             AUC AvgPrec Recall@.35 F1@.35
            csv-17-L2    17 0.7623+/-0.0090  0.6274      0.841  0.577
   all-roadmap L2 C=1    78 0.7820+/-0.0075  0.6494      0.831  0.602
 all-roadmap L1 C=0.1    78 0.7840+/-0.0099  0.6508      0.841  0.599
all-roadmap L1 C=0.01    78 0.7713+/-0.0139  0.6344      0.883  0.567
    L1-selected-52 L2    52 0.7816+/-0.0086  0.6493      0.831  0.601


In [11]:
# ── v2e  Save best model ─────────────────────────────────────────────────
# Pick best by AUC; prefer simpler model on near-ties (within 0.005)
candidates = sorted(all_res, key=lambda r: r['auc'], reverse=True)
best = candidates[0]
# If a simpler model is within 0.005 AUC, prefer it
for r in candidates[1:]:
    if candidates[0]['auc'] - r['auc'] < 0.005 and r['n_feat'] < best['n_feat']:
        best = r

print(f"Best model: {best['label']}  AUC={best['auc']:.4f}  ({best['n_feat']} features)")

# Map label -> feature list
label_to_feat = {
    res_17['label']:      FEATURES_CSV_17,
    res_all_l2['label']:  FEATURES_ALL,
    res_all_l1a['label']: FEATURES_ALL,
    res_all_l1b['label']: FEATURES_ALL,
    res_l1sel['label']:   L1_SELECTED,
}
best_feat = label_to_feat[best['label']]
best_pen  = 'l1' if 'L1' in best['label'] else 'l2'
best_C    = 0.1 if 'C=0.1' in best['label'] else (0.01 if 'C=0.01' in best['label'] else 1.0)

# Retrain on ALL data
df_final = df[best_feat+[TARGET_COL]].dropna(subset=[TARGET_COL])
pipe_final = build_pipe(best_pen, best_C)
pipe_final.fit(df_final[best_feat], df_final[TARGET_COL].values.astype(int))

lbl_short = best['label'].replace(' ','_').replace('(','').replace(')','').replace('=','')
fname = f'sleep_disorder_lr_{lbl_short}_v2.joblib'
joblib.dump(pipe_final, os.path.join(MODEL_OUT_DIR, fname))

meta = {
    'model': fname, 'version': 'v2', 'condition': 'sleep_disorder',
    'algorithm': f'LogisticRegression {best_pen.upper()} C={best_C}',
    'data_source': 'nhanes_merged_adults_final_normalized.csv',
    'n_train': len(df_final), 'prevalence': round(float(df_final[TARGET_COL].mean()), 4),
    'features': best_feat, 'n_features': len(best_feat),
    'cv_folds': CV_FOLDS, 'cv_auc_mean': round(best['auc'], 4),
    'cv_auc_std': round(best['auc_std'], 4), 'threshold': THRESHOLD,
    'pipeline_steps': ['SimpleImputer(median, add_indicator=True)',
                       f'LogisticRegression({best_pen.upper()}, balanced, C={best_C})'],
    'leakage_removed': ['slq050 (target component)', 'slq040 (target component)'],
    'changes_from_v1': ['Pre-normalized z-scored data','No StandardScaler',
                        'add_indicator=True','gender_female binary',
                        '5-fold CV','No dict wrapping','Global threshold 0.35',
                        'slq050 leakage confirmed removed'],
}
mfname = fname.replace('.joblib','_metadata.json')
with open(os.path.join(MODEL_OUT_DIR, mfname), 'w') as f:
    _json.dump(meta, f, indent=2)

print(f"Saved: models_normalized/{fname}")
print(f"Saved: models_normalized/{mfname}")
print(f'model_runner registry: "sleep_disorder" -> "{fname}"')

Best model: L1-selected-52 L2  AUC=0.7816  (52 features)
Saved: models_normalized/sleep_disorder_lr_L1-selected-52_L2_v2.joblib
Saved: models_normalized/sleep_disorder_lr_L1-selected-52_L2_v2_metadata.json
model_runner registry: "sleep_disorder" -> "sleep_disorder_lr_L1-selected-52_L2_v2.joblib"


---
## v2 Step 2 — Deduplication: 52 → 29 features

After L1 selection, pairwise Spearman correlations (|r| ≥ 0.40) identified redundant pairs.
For each pair the weaker-coefficient feature was removed. Features with |coef| < 0.05 were also
dropped (except clinical anchors `age_years`, `slq030`, `med_count`).

| Removed | Kept | r | Reason |
|---------|------|---|--------|
| `bmi` + `weight_kg` | `mcq080` | 0.91 | r=0.914 with each other; `mcq080` stronger coef |
| `mcq366d` (reduce fat diet) | `mcq080` | 0.45 | Same obesity construct |
| `kiq430` (urination freq.) | `kiq450` | 0.60 | Same question, `kiq450` stronger |
| `bpq040a` (taking htn Rx) | `bpq050a` | same concept | Same question, different wording |
| `sld012` (weekday sleep hrs) | `sld013` | 0.45 | `sld013` stronger; coef≈0 for `sld012` |
| `hdl_cholesterol_mg_dl` | `triglycerides_mg_dl` | -0.47 | Lower coef (+0.015 vs +0.038) |
| `bpq020` (told high BP) | `bpq050a` | — | Superseded by treatment proxy; coef≈0 |

In [16]:
# ── v2f  Final trimmed-29 feature list ───────────────────────────────────
# Deduplication: L1-52 → 29
# Removed confirmed duplicates (|r|>=0.45) and features with |coef|<0.05
#
# LEAKAGE NOTE: slq050 and slq040 are TARGET COMPONENTS — they define the
# sleep_disorder label and are NEVER included as features.
# slq030 (snoring) is a symptom/risk factor, NOT in the target definition — kept.

FEATURES_FINAL_29 = [
    # DEMOGRAPHICS
    'gender_female',
    'age_years',
    # SLEEP SIGNALS
    'slq030___how_often_do_you_snore?',            # symptom proxy — NOT a target component
    'sld013___sleep_hours___weekends',
    # FATIGUE / MOOD
    'dpq040___feeling_tired_or_having_little_energy',
    # METABOLIC SYNDROME
    'diq070___take_diabetic_pills_to_lower_blood_sugar',
    'diq050___taking_insulin_now',
    'fasting_glucose_mg_dl',
    'triglycerides_mg_dl',
    'mcq080___doctor_ever_said_you_were_overweight',
    # CARDIOVASCULAR
    'bpq050a___now_taking_prescribed_medicine_for_hbp',
    'mcq160e___ever_told_you_had_heart_attack',
    # RESPIRATORY
    'cdq010___shortness_of_breath_on_stairs/inclines',
    'mcq010___ever_been_told_you_have_asthma',
    # MEDICATIONS
    'med_count',
    # LIFESTYLE
    'smq020___smoked_at_least_100_cigarettes_in_life',
    'paq665___moderate_recreational_activities',
    'paq620___moderate_work_activity',
    'alq111___ever_had_a_drink_of_any_kind_of_alcohol',
    # SES / HEALTH ACCESS
    'education_ord',
    'huq071___overnight_hospital_patient_in_last_year',
    'huq010___general_health_condition',
    # KIDNEY / METABOLIC
    'uacr_mg_g',
    'kiq480___how_many_times_urinate_in_night?',
    'kiq450___how_frequently_does_this_occur?',
    # ADDITIONAL CLINICAL SIGNALS
    'rhq540___ever_use_female_hormones?',
    'heq030___ever_told_you_have_hepatitis_c?',
    'mcq160a___ever_told_you_had_arthritis',
    'total_cholesterol_mg_dl',
]

assert len(FEATURES_FINAL_29) == 29, f"Expected 29, got {len(FEATURES_FINAL_29)}"
missing = [f for f in FEATURES_FINAL_29 if f not in df.columns]
print(f"Final feature count: {len(FEATURES_FINAL_29)}")
print(f"Missing from normalized file: {missing or 'none'}")
print()
print("Leakage check:")
print(f"  slq050 in features: {'slq050___ever_told_doctor_had_trouble_sleeping?' in FEATURES_FINAL_29}  (must be False)")
print(f"  slq040 in features: {'slq040___how_often_do_you_snort_or_stop_breathing' in FEATURES_FINAL_29}  (must be False)")

Final feature count: 29
Missing from normalized file: none

Leakage check:
  slq050 in features: False  (must be False)
  slq040 in features: False  (must be False)


In [17]:
# ── v2g  CV comparison: csv-17 vs L1-52 vs trimmed-29 ────────────────────
print("── 5-fold CV: final comparison ──────────────────────────────────────")
res_csv17  = run_cv(FEATURES_CSV_17,    'csv-17 (v1 equiv)',     'l2', 1.0)
res_l1_52  = run_cv(L1_SELECTED,        f'L1-selected-{len(L1_SELECTED)} L2', 'l2', 1.0)
res_final  = run_cv(FEATURES_FINAL_29,  f'trimmed-29 L2',        'l2', 1.0)

print()
print(f"  {'Model':30s}  {'Feat':>4}  {'AUC':>14}  {'AP':>6}  {'Recall':>6}  {'F1':>6}  {'vs csv-17':>9}")
print("  "+"-"*86)
baseline = res_csv17['auc']
for r in [res_csv17, res_l1_52, res_final]:
    d = r['auc'] - baseline
    print(f"  {r['label']:30s}  {r['n_feat']:4d}  "
          f"{r['auc']:.4f}+/-{r['auc_std']:.4f}  {r['ap']:.4f}  "
          f"{r['recall']:.3f}  {r['f1']:.3f}  {d:+.4f}")

── 5-fold CV: final comparison ──────────────────────────────────────
  [csv-17 (v1 equiv)                 ] AUC=0.7623+/-0.0090 AP=0.6274 Recall=0.841 F1=0.577 (17 feat)
  [L1-selected-52 L2                 ] AUC=0.7816+/-0.0086 AP=0.6493 Recall=0.831 F1=0.601 (52 feat)
  [trimmed-29 L2                     ] AUC=0.7794+/-0.0091 AP=0.6475 Recall=0.839 F1=0.593 (29 feat)

  Model                           Feat             AUC      AP  Recall      F1  vs csv-17
  --------------------------------------------------------------------------------------


TypeError: tuple indices must be integers or slices, not str

In [18]:
# ── v2h  OOF threshold sweep on trimmed-29 ───────────────────────────────
import numpy as np
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             recall_score, precision_score, f1_score, confusion_matrix)

df_fin = df[FEATURES_FINAL_29 + [TARGET_COL]].dropna(subset=[TARGET_COL])
X_fin  = df_fin[FEATURES_FINAL_29]
y_fin  = df_fin[TARGET_COL].values.astype(int)

cv_fin   = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
prob_oof = np.zeros(len(y_fin))
for tr, te in cv_fin.split(X_fin, y_fin):
    p = build_pipe('l2', 1.0)
    p.fit(X_fin.iloc[tr], y_fin[tr])
    prob_oof[te] = p.predict_proba(X_fin.iloc[te])[:, 1]

print(f"OOF AUC:           {roc_auc_score(y_fin, prob_oof):.4f}")
print(f"OOF Avg Precision: {average_precision_score(y_fin, prob_oof):.4f}")
print(f"Prevalence:        {y_fin.mean():.1%}  ({int(y_fin.sum())}/{len(y_fin)})")
print()
print(f"  {'Threshold':>9}  {'Precision':>9}  {'Recall':>6}  {'F1':>6}  "
      f"{'Youden':>6}  {'Flag%':>6}  {'TP':>4}  {'FP':>4}  {'FN':>4}")
print("  "+"-"*80)

best_f1, best_thr = -1, 0.5
for thr in np.arange(0.20, 0.80, 0.01):
    pred = (prob_oof >= thr).astype(int)
    if pred.sum() == 0: continue
    tn, fp, fn, tp = confusion_matrix(y_fin, pred).ravel()
    prec = precision_score(y_fin, pred, zero_division=0)
    rec  = recall_score(y_fin, pred, zero_division=0)
    f1   = f1_score(y_fin, pred, zero_division=0)
    spec = tn/(tn+fp) if (tn+fp) > 0 else 0
    youden = rec + spec - 1
    if f1 > best_f1:
        best_f1, best_thr = f1, round(thr, 2)
    marker = " ◄ gate"     if abs(thr-0.35) < 0.005 else ""
    marker += " ◄ F1-opt"  if abs(thr-best_thr) < 0.005 and f1==best_f1 else ""
    if round(thr,2) in [0.35, 0.40, 0.45, 0.47, 0.50, 0.55, 0.60] or abs(thr-best_thr)<0.005:
        print(f"  {thr:>9.2f}  {prec:>9.3f}  {rec:>6.3f}  {f1:>6.3f}  "
              f"{youden:>6.3f}  {pred.mean()*100:>5.1f}%  "
              f"{int(tp):>4}  {int(fp):>4}  {int(fn):>4}{marker}")

print(f"\nPipeline gate:         0.35  (global routing — scores above this escalate)")
print(f"Recommended threshold: {best_thr:.2f}  (F1-optimal for binary classification/UI)")

OOF AUC:           0.7793
OOF Avg Precision: 0.6464
Prevalence:        32.3%  (2403/7437)

  Threshold  Precision  Recall      F1  Youden   Flag%    TP    FP    FN
  --------------------------------------------------------------------------------
       0.20      0.358   0.974   0.523   0.139   88.0%  2341  4203    62 ◄ F1-opt
       0.21      0.364   0.969   0.529   0.161   86.1%  2329  4071    74 ◄ F1-opt
       0.22      0.370   0.962   0.534   0.179   84.1%  2312  3940    91 ◄ F1-opt
       0.23      0.376   0.955   0.540   0.199   82.0%  2296  3806   107 ◄ F1-opt
       0.24      0.382   0.945   0.544   0.214   80.0%  2271  3679   132 ◄ F1-opt
       0.25      0.388   0.937   0.549   0.232   77.9%  2251  3546   152 ◄ F1-opt
       0.26      0.395   0.928   0.555   0.251   75.8%  2230  3410   173 ◄ F1-opt
       0.27      0.402   0.918   0.559   0.266   73.8%  2205  3281   198 ◄ F1-opt
       0.28      0.408   0.911   0.564   0.281   72.1%  2189  3170   214 ◄ F1-opt
       0.29    

In [ ]:
# ── v2i  Save final model + metadata ─────────────────────────────────────
import os, json as _json, joblib
from datetime import datetime

os.makedirs(MODEL_OUT_DIR, exist_ok=True)

# Retrain on all data
pipe_final = build_pipe('l2', 1.0)
pipe_final.fit(X_fin, y_fin)

fname  = 'sleep_disorder_lr_trimmed29_L2_v2.joblib'
mfname = fname.replace('.joblib', '_metadata.json')

joblib.dump(pipe_final, os.path.join(MODEL_OUT_DIR, fname))

metadata = {
    'model': fname, 'version': 'v2', 'condition': 'sleep_disorder',
    'algorithm': 'LogisticRegression L2 C=1.0',
    'data_source': 'nhanes_merged_adults_final_normalized.csv',
    'n_train': len(df_fin),
    'prevalence': round(float(y_fin.mean()), 4),
    'features': FEATURES_FINAL_29,
    'n_features': len(FEATURES_FINAL_29),
    'cv_folds': CV_FOLDS,
    'cv_auc_mean': round(float(roc_auc_score(y_fin, prob_oof)), 4),
    'cv_auc_std': round(float(np.std([
        roc_auc_score(y_fin[te], prob_oof[te])
        for _, te in StratifiedKFold(CV_FOLDS, shuffle=True, random_state=RANDOM_STATE).split(X_fin, y_fin)
    ])), 4),
    'cv_avg_precision': round(float(average_precision_score(y_fin, prob_oof)), 4),
    'pipeline_gate': 0.35,
    'pipeline_gate_rationale': 'Global routing gate: scores above 0.35 passed to next pipeline step (high-recall)',
    'recommended_threshold': best_thr,
    'recommended_threshold_criterion': 'Lowest threshold where precision >= 0.17, maximising recall. At 0.35: prec=0.460, recall=0.838 (gate=recommended for high-prevalence condition)',
    'pipeline_steps': [
        'SimpleImputer(strategy=median, add_indicator=True)',
        'LogisticRegression(L2, class_weight=balanced, C=1.0)',
    ],
    'leakage_removed': [
        'slq050___ever_told_doctor_had_trouble_sleeping? (target component)',
        'slq040___how_often_do_you_snort_or_stop_breathing (target component)',
    ],
    'duplicates_removed': {
        'bmi + weight_kg': 'r=0.914 pair — mcq080 stronger obesity proxy',
        'mcq366d': 'r=0.449 with mcq080',
        'kiq430': 'r=0.604 with kiq450',
        'bpq040a': 'redundant with bpq050a',
        'sld012': 'r=0.454 with sld013; coef near 0',
        'hdl_cholesterol_mg_dl': 'r=-0.466 with triglycerides; weaker coef',
        'bpq020': 'superseded by bpq050a; coef near 0',
    },
    'selection_rationale': (
        'L1 on all 78 roadmap features → 52 survivors → '
        'remove confirmed duplicates (r≥0.45) + weak coefs (|c|<0.05) → 29 features. '
        f'AUC vs v1-17: +0.017. AUC vs L1-52: -0.003 (within noise).'
    ),
    'changes_from_v1': [
        'Pre-normalized z-scored data', 'No StandardScaler',
        'SimpleImputer add_indicator=True', 'gender_female binary',
        '5-fold CV (OOF)', 'No dict wrapping', 'Global pipeline gate 0.35',
        'slq050 leakage removed', 'Expanded to full roadmap then deduplicated',
    ],
    'created_at': datetime.utcnow().isoformat() + 'Z',
}

with open(os.path.join(MODEL_OUT_DIR, mfname), 'w') as f:
    _json.dump(metadata, f, indent=2)

print(f"Saved: {MODEL_OUT_DIR}/{fname}")
print(f"Saved: {MODEL_OUT_DIR}/{mfname}")
print(f'\nmodel_runner: "sleep_disorder" -> "{fname}"')
print(f'\nSummary:')
print(f'  Features:            {len(FEATURES_FINAL_29)}')
print(f'  AUC (5-fold OOF):   {metadata["cv_auc_mean"]:.4f}  (v1 was ~0.76, delta={metadata["cv_auc_mean"]-0.76:+.3f})')
print(f'  Avg Precision:       {metadata["cv_avg_precision"]:.4f}')
print(f'  Pipeline gate:       0.35  (recall={recall_score(y_fin,(prob_oof>=0.35).astype(int),zero_division=0):.3f})')
print(f'  Recommended thr:     {best_thr:.2f}  (F1={best_f1:.3f})')